# Workday Sales RAG Model Deployment & Serving

This notebook handles model serving endpoint creation, workload selection, authentication configuration (Service Principal OAuth + PAT), readiness checks, endpoint testing, and deployment best practices for higher environments.

In [0]:
notedbutils.widgets.text("workload_type", "CPU")
dbutils.widgets.text("workload_size", "Small")
dbutils.widgets.text("scale_to_zero_enabled", "true")
dbutils.widgets.text("registered_model_name", "rag_agentic.workday_demos.workday_sales_rag")
dbutils.widgets.text("model_version", "1")
dbutils.widgets.text("model_name", "workday_sales_rag")
dbutils.widgets.text("llm_endpoint_name", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.text("vs_endpoint_name", "sales-endpoint-rag_agentic")
dbutils.widgets.text("customer_feedback_index", "rag_agentic.workday_demos.customer_feedback_index")
dbutils.widgets.text("meeting_notes_index", "rag_agentic.workday_demos.meeting_notes_index")
dbutils.widgets.text("email_communications_index", "rag_agentic.workday_demos.email_communications_index")
dbutils.widgets.text("secret_scope", "agentic")

registered_model_name = dbutils.widgets.get("registered_model_name")
model_version = dbutils.widgets.get("model_version")
model_name = dbutils.widgets.get("model_name")
workload_type = dbutils.widgets.get("workload_type")
workload_size = dbutils.widgets.get("workload_size")
scale_to_zero_enabled = dbutils.widgets.get("scale_to_zero_enabled").lower() == "true"
llm_endpoint_name = dbutils.widgets.get("llm_endpoint_name")
vs_endpoint_name = dbutils.widgets.get("vs_endpoint_name")
customer_feedback_index = dbutils.widgets.get("customer_feedback_index")
meeting_notes_index = dbutils.widgets.get("meeting_notes_index")
email_communications_index = dbutils.widgets.get("email_communications_index")
secret_scope = dbutils.widgets.get("secret_scope")

In [0]:
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType

In [0]:
from databricks.sdk import WorkspaceClient

# Initialize Databricks SDK client
w = WorkspaceClient()

print("✅ Databricks Workspace Client initialized")
print(f"   Host: {w.config.host}")

In [0]:
# Endpoint configuration
endpoint_name = f"{model_name}_endpoint"  # workday_sales_rag_endpoint

print(f"\n🚀 Endpoint Deployment Plan")
print("=" * 60)
print(f"   Endpoint Name: {endpoint_name}")
print(f"   Model: {registered_model_name}")
print(f"   Version: {model_version}")
print(f"   Workload: {workload_type} / {workload_size}")
print(f"   Scale to Zero: {scale_to_zero_enabled}")
print("\n⏱️  Deployment typically takes ~15 minutes")
print("=" * 60)

# Build served entity configuration with authentication
secret_scope = dbutils.widgets.get("secret_scope")
desired_env_vars = {
    "DATABRICKS_HOST": w.config.host if w.config.host.startswith('http') else f"https://{w.config.host}",
    "DATABRICKS_CLIENT_ID": f"{{{{secrets/{secret_scope}/sp_client_id}}}}",
    "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{secret_scope}/sp_client_secret}}}}",
    "LLM_ENDPOINT_NAME": dbutils.widgets.get("llm_endpoint_name"),
    "VS_ENDPOINT_NAME": dbutils.widgets.get("vs_endpoint_name"),
    "CUSTOMER_FEEDBACK_INDEX": dbutils.widgets.get("customer_feedback_index"),
    "MEETING_NOTES_INDEX": dbutils.widgets.get("meeting_notes_index"),
    "EMAIL_COMMUNICATIONS_INDEX": dbutils.widgets.get("email_communications_index"),
    "NUM_RESULTS": "5",
    "TEMPERATURE": "0.1",
    "MAX_TOKENS": "1000",
}

desired_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=str(model_version),
    workload_type=ServingModelWorkloadType(workload_type),
    workload_size=workload_size,
    scale_to_zero_enabled=scale_to_zero_enabled,
    environment_vars=desired_env_vars,
)

def needs_update(current_entity, desired_entity) -> tuple[bool, list[str]]:
    """Check if endpoint configuration needs updating."""
    changes = []
    
    # Check model version
    if current_entity.entity_version != desired_entity.entity_version:
        changes.append(f"Version: {current_entity.entity_version} → {desired_entity.entity_version}")
    
    # Check workload configuration
    if current_entity.workload_size != desired_entity.workload_size:
        changes.append(f"Workload Size: {current_entity.workload_size} → {desired_entity.workload_size}")
    
    if current_entity.scale_to_zero_enabled != desired_entity.scale_to_zero_enabled:
        changes.append(f"Scale to Zero: {current_entity.scale_to_zero_enabled} → {desired_entity.scale_to_zero_enabled}")
    
    # Check environment variables (key comparison only, not secret values)
    current_env_keys = set(current_entity.environment_vars.keys()) if current_entity.environment_vars else set()
    desired_env_keys = set(desired_entity.environment_vars.keys())
    
    if current_env_keys != desired_env_keys:
        missing = desired_env_keys - current_env_keys
        extra = current_env_keys - desired_env_keys
        if missing:
            changes.append(f"Missing env vars: {missing}")
        if extra:
            changes.append(f"Extra env vars: {extra}")
    
    return len(changes) > 0, changes

# Check if endpoint already exists
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"\n🔍 Endpoint '{endpoint_name}' exists. Checking configuration...")
    
    # Get current configuration
    current_entities = list(
        existing_endpoint.config.served_entities
        if existing_endpoint.config and existing_endpoint.config.served_entities
        else []
    )
    
    if len(current_entities) == 0:
        raise ValueError(f"Endpoint exists but has no served entities")
    
    current_entity = current_entities[0]
    
    # Validate model name matches
    if current_entity.entity_name != registered_model_name:
        print(f"\n⚠️  ERROR: Endpoint currently serves: {current_entity.entity_name}")
        print(f"   Cannot replace with: {registered_model_name}")
        print(f"\n👉 Choose a different endpoint name or delete the existing endpoint.")
        raise ValueError(
            f"Endpoint name collision: '{endpoint_name}' already serves '{current_entity.entity_name}'"
        )
    
    # Check if update is needed (idempotency check)
    update_needed, changes = needs_update(current_entity, desired_entity)
    
    if not update_needed:
        print(f"\n✅ Endpoint configuration is already up-to-date!")
        print(f"   Model: {registered_model_name}")
        print(f"   Version: {model_version}")
        print(f"   Workload: {workload_type} / {workload_size}")
        print(f"   Scale to Zero: {scale_to_zero_enabled}")
        print(f"\n💡 No changes needed. Endpoint is ready to use.")
        action_taken = "no_change"
    else:
        print(f"\n🔄 Configuration changes detected:")
        for change in changes:
            print(f"   • {change}")
        
        print(f"\n📤 Updating endpoint configuration...")
        w.serving_endpoints.update_config(
            name=endpoint_name,
            served_entities=[desired_entity],
        )
        print(f"\n✅ Endpoint update initiated!")
        action_taken = "updated"
    
except NotFound:
    # Create new endpoint
    print(f"\n✨ Endpoint '{endpoint_name}' does not exist. Creating...")
    endpoint_config = EndpointCoreConfigInput(served_entities=[desired_entity])
    w.serving_endpoints.create(
        name=endpoint_name,
        config=endpoint_config,
    )
    print(f"\n✅ Endpoint creation initiated!")
    action_taken = "created"

except Exception as e:
    print(f"\n❌ Error during endpoint deployment: {str(e)}")
    raise

# Summary
print(f"\n" + "=" * 60)
print(f"📊 Deployment Summary")
print("=" * 60)
print(f"   Endpoint: {endpoint_name}")
print(f"   Action: {action_taken.upper().replace('_', ' ')}")
print(f"   Model: {registered_model_name} v{model_version}")
print(f"\n🔗 Endpoint URL:")
print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")

if action_taken in ["created", "updated"]:
    print(f"\n⏳ Proceeding to readiness check...")
else:
    print(f"\n✅ Endpoint is ready to use (no deployment needed)")